# RTA Report — Optimized Version

In [55]:
import pandas as pd
import glob
import numpy as np
from datetime import datetime, timedelta, date
from pathlib import Path
import polars as pl
import os
import re
import shutil

In [56]:
# ── HELPER FUNCTIONS ─────────────────────────────────────────────────────────

def input_data(folder_path: str, sheet_name=None) -> pl.DataFrame:
    """Read all xlsx/csv in a folder into one Polars DataFrame (all-String schema)."""
    file_paths = glob.glob(f"{folder_path}/*.xlsx") + glob.glob(f"{folder_path}/*.csv")
    df_list = []
    for file in file_paths:
        try:
            if file.endswith('.xlsx'):
                df = pl.read_excel(file, sheet_name=sheet_name)
            else:
                try:
                    df = pl.read_csv(file, encoding='utf-8')
                except Exception:
                    df = pl.read_csv(file, encoding='ISO-8859-1', ignore_errors=True)
            df = df.with_columns([pl.col(c).cast(pl.String) for c in df.columns])
            df_list.append(df)
        except Exception as e:
            print(f"Error reading {file}: {e}")
    if not df_list:
        return pl.DataFrame()
    return pl.concat(df_list, how='diagonal')


# Vectorised datetime parser -- replaces the row-wise map_elements(convert_datetime)
_DATE_FORMATS = [
    "%Y-%m-%d %H:%M:%S%.f",
    "%Y-%m-%d %H:%M:%S",
    "%m/%d/%Y %H:%M:%S",
    "%m/%d/%Y %H:%M",
    "%m-%d-%Y %H:%M",
    "%m/%d/%Y",
    "%Y-%m-%d",
]

def parse_datetime_col(series: pl.Series) -> pl.Series:
    """
    Try each format; keep the one yielding most non-nulls.
    Falls back to pandas mixed parsing for non-zero-padded formats (e.g. 4/6/2026 0:00).
    Returns a Utf8 Series formatted as '%Y-%m-%d %H:%M:%S'.
    """
    best, best_count = None, -1
    for fmt in _DATE_FORMATS:
        parsed = series.str.strptime(pl.Datetime, fmt, strict=False)
        count  = parsed.drop_nulls().len()
        if count > best_count:
            best, best_count = parsed, count
        if count == series.len():
            break

    # Fallback: use pandas mixed parser for remaining nulls
    # Handles non-zero-padded formats like "4/6/2026 0:00" on all platforms
    null_mask = best.is_null()
    if null_mask.any():
        null_values = series.filter(null_mask).to_list()

        import pandas as _pd
        repaired_pd = _pd.to_datetime(null_values, errors='coerce', dayfirst=False)
        repaired_strs = [
            v.strftime("%Y-%m-%d %H:%M:%S") if not _pd.isna(v) else None
            for v in repaired_pd
        ]
        repaired_dt = pl.Series(repaired_strs).str.strptime(
            pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False
        )

        # Scatter repaired values back to original positions
        orig_idx = null_mask.arg_true().to_list()
        new_vals = best.to_list()
        for i, v in zip(orig_idx, repaired_dt.to_list()):
            new_vals[i] = v
        best = pl.Series(new_vals, dtype=pl.Datetime)

    return best.dt.strftime("%Y-%m-%d %H:%M:%S")

def copy_data_files(source_folder: str, destination_folder: str) -> None:
    os.makedirs(destination_folder, exist_ok=True)

    csv_files = glob.glob(os.path.join(source_folder, "*.csv"))
    xlsx_files = glob.glob(os.path.join(source_folder, "*.xlsx"))
    files = csv_files + xlsx_files
    
    for f in files:
        shutil.copy2(f, destination_folder)

def restore_from_storage(paths_dict: dict, periods: list, mapping: dict) -> int:
    total_restored = 0
    for storage_key, input_key in mapping.items():
        source_dir = paths_dict.get(storage_key)
        dest_dir = paths_dict.get(input_key)

        if not source_dir or not dest_dir or not os.path.exists(source_dir):
            continue

        os.makedirs(dest_dir, exist_ok=True)

        for period in periods:
            flex_period = period.replace('-', '?').replace('_', '?')
            
            csv_query = os.path.join(source_dir, f"*{flex_period}*.csv")
            xlsx_query = os.path.join(source_dir, f"*{flex_period}*.xlsx")
            
            matched_files = glob.glob(csv_query) + glob.glob(xlsx_query)

            for filepath in matched_files:
                try:
                    shutil.copy2(filepath, dest_dir)
                    total_restored += 1
                except Exception:
                    pass
                    
    return total_restored

def clean_input_folders(paths_dict: dict, periods: list, input_keys: list) -> int:
    total_deleted = 0
    for input_key in input_keys:
        target_dir = paths_dict.get(input_key)
        
        if not target_dir or not os.path.exists(target_dir):
            continue

        for period in periods:
            flex_period = period.replace('-', '?').replace('_', '?')
            
            csv_query = os.path.join(target_dir, f"*{flex_period}*.csv")
            xlsx_query = os.path.join(target_dir, f"*{flex_period}*.xlsx")
            
            matched_files = glob.glob(csv_query) + glob.glob(xlsx_query)

            for filepath in matched_files:
                try:
                    os.remove(filepath)
                    total_deleted += 1
                except Exception:
                    pass
                    
    return total_deleted

today = datetime.today().strftime('%b_%d_%Y')

In [57]:
# ── PATHS ────────────────────────────────────────────────────────────────────

first_glob = os.path.expanduser('~').replace('\\', '/')
test_path  = f'{first_glob}/Concentrix Corporation'
if not os.path.exists(test_path):
    raise FileNotFoundError(f'Not found the path: {test_path}')

folder_paths = {
    'input_iex_rawdata'             : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_AGENT_IEX_FOR_REPORT',
    'storage_iex_rawdata'           : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_INPUT_AGENT_IEX',
    'input_agent_activity'          : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_AGENT_ACTIVITY_FOR_REPORT',
    'storage_agent_activity'        : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_INPUT_AGENT_ACTIVITY',
    'input_iex'                     : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_FOR_REPORT',
    'input_iex_intervals'           : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_INTERVALS',
    'input_pull_out'                : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Save/EXP Leave Management.xlsx',
    'input_ramco_code'              : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_RAMCO_CODE',
    'input_ramco_ot'                : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_OT_RAMCO_CODE',
    'output_rta'                    : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_RTA_FOR_REPORT',
    'output_rta_all'                : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_RTA',
    'rta_intervals_output'          : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_ACTIVITY_INTERVALS',
    'ramco_output'                  : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata',
    'hc_extend_by_month'            : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month',
    'resources'                     : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources',
}

print('--- FULL FOLDER PATHS LIST ---')
for k, v in folder_paths.items():
    print(f'{k}: {v}')

--- FULL FOLDER PATHS LIST ---
input_iex_rawdata: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_AGENT_IEX_FOR_REPORT
storage_iex_rawdata: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_INPUT_AGENT_IEX
input_agent_activity: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_AGENT_ACTIVITY_FOR_REPORT
storage_agent_activity: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_INPUT_AGENT_ACTIVITY
input_iex: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_FOR_REPORT
input_iex_intervals: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_INTERVALS
input_pull_out: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Save/EXP Leave Management.xlsx
input_ramco_code: C:/Users/ADMIN/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Raw

In [ ]:
copy_mapping = {
    "input_iex_rawdata": "storage_iex_rawdata",
    "input_agent_activity": "storage_agent_activity"
}

for src_key, dest_key in copy_mapping.items():
    src_folder = folder_paths[src_key]
    dest_folder = folder_paths[dest_key]
    
    copy_data_files(src_folder, dest_folder)

target_periods = ["2026-01","2026-02","2026-03","2026-04"]

restore_mapping = {
    'storage_iex_rawdata': 'input_iex_rawdata',
    'storage_agent_activity': 'input_agent_activity'
}

keys_to_clean = ['input_iex_rawdata', 'input_agent_activity']
# restored_count = restore_from_storage(folder_paths, target_periods, restore_mapping)
# deleted_count = clean_input_folders(folder_paths, target_periods, keys_to_clean)

In [51]:
# ── LOAD & CLEAN AGENT ACTIVITY ───────────────────────────────────────────────

KEEP_COLS = [
    'Agent Email', 'Supervisor Email', 'Agent Queue Group Name', 'Agent State',
    'Number of Active Contacts', 'Productive Aux Flag (Yes / No)',
    'Start Time', 'End Time', 'Total Time in seconds', 'Agent Business Location',
]

_raw = input_data(folder_paths['input_agent_activity'])
AGENT_ACTIVITY_INPUT = (
    _raw
    .select([c for c in KEEP_COLS if c in _raw.columns])
    .unique()
    .with_columns(
        pl.col('Total Time in seconds').str.replace_all(',', '').cast(pl.Float64)
    )
    .filter(pl.col('Total Time in seconds') < 3600 * 10)
)
# ── LOAD IEX ──────────────────────────────────────────────────────────────────

COLUMNS_TO_SEC = ['Time_Of_Day', 'Open Time', 'Extra Time', 'Break Time',
                  'Lunch Time', 'Training', 'NCNS', 'AL', 'Target']

IEX_DT_COLS = ['Datetime_Fluctuate_Start_Shift', 'Datetime_Fluctuate_End_Shift',
               'Datetime_First_Start_Shift', 'Datetime_First_End_Shift']

IEX = (
    input_data(folder_paths['input_iex'])
    .unique()
    .with_columns([
        pl.col('Date').str.to_date('%Y-%m-%d', strict=False),
        *[
            pl.coalesce([
                pl.col(c).str.to_datetime('%Y-%m-%d %H:%M:%S%.f', strict=False),
                pl.col(c).str.to_datetime('%Y-%m-%d %H:%M:%S', strict=False),
                pl.col(c).str.to_datetime('%m/%d/%Y %H:%M', strict=False),
                pl.col(c).str.to_datetime(strict=False)
            ]).alias(c) for c in IEX_DT_COLS
        ],
        pl.col(['Night_Shift', 'Target', 'Unplanned', 'Planned',
                'Roster Presented', 'Roster Scheduled']).cast(pl.Float64),
    ])
    .with_columns([
        pl.when(pl.col(c).cast(pl.Float64, strict=False).fill_null(0) <= 100)
          .then(pl.col(c).cast(pl.Float64, strict=False).fill_null(0) * 3600.0)
          .otherwise(pl.col(c).cast(pl.Float64, strict=False).fill_null(0))
          .alias(c)
        for c in COLUMNS_TO_SEC
    ])
)

IEX = IEX.with_columns(
    pl.when(
        (pl.col("Datetime_First_End_Shift").dt.date() > pl.col("Date")) & 
        (pl.col("Datetime_First_End_Shift").dt.hour() >= 0) & 
        (pl.col("Datetime_First_End_Shift").dt.hour() <= 10)
    )
    .then(1.0)
    .otherwise(pl.col("Night_Shift"))
    .alias("Night_Shift")
)
# ── NIGHT SHIFT LOOKUP (FIXED) ────────────────────────────────────────────────

Night_Shift_1 = IEX[['Date', 'Email Id', 'Night_Shift']].unique()

# Tạo bảng Previous Night Shift bằng cách đẩy Date lên 1 ngày để mapping đúng với hôm sau
Prev_NS = (
    Night_Shift_1
    .with_columns((pl.col('Date') + pl.duration(days=1)).alias('Date'))
    .rename({'Night_Shift': 'Previous_Night_Shift'})
)

# ── PULL OUT ──────────────────────────────────────────────────────────────────

PULL_OUT = (
    pl.read_excel(folder_paths['input_pull_out'], sheet_name='Pull Out')
    .unique()
    .with_columns(
        (pl.col('Pull-out duration (mins)').cast(pl.Float64) / 60).alias('Duration')
    )
    .group_by(['Pull-out Date', 'Emp ID'])
    .agg(pl.col('Duration').sum().cast(pl.Float64).alias('total_time_pull_out'))
)

# ── HC MASTER DATABASE ────────────────────────────────────────────────────────

HC_MASTER_DATABASE = (
    input_data(folder_paths['hc_extend_by_month'])
    .rename({'Date Start Week': 'Week_Monday'})
    .with_columns(pl.col('Date').str.to_date('%Y-%m-%d', strict=False))
)

Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 43, falling back to string
Could not determine dtype for column 8, falling back to string


In [52]:
# ── RAMCO PROCESSING ──────────────────────────────────────────────────────────

def process_all_ramco_data(folder_path: str, hc_db: pl.DataFrame):
    """Parse Attendance and OT RAMCO files; return (RAMCO, RAMCO_OT)."""

    def get_unique_cols(df_cols, header_row, is_ot=False):
        new_cols, counts = [], {}
        for idx, col in enumerate(df_cols):
            is_day = col.startswith('day_') if is_ot else col.isdigit()
            if is_day:
                raw  = header_row[idx]
                base = str(raw).strip() if raw not in [None, ''] else f'empty_{idx}'
            else:
                base = col
            if base in counts:
                counts[base] += 1
                new_cols.append(f'{base}_{counts[base]}')
            else:
                counts[base] = 0
                new_cols.append(base)
        return new_cols

    def extract_ot_details(df: pl.DataFrame) -> pl.DataFrame:
        return df.with_columns([
            pl.col('Temp').str.extract(r'\((\d+\.?\d*)\s*hrs', 1).cast(pl.Float64).alias('Hours'),
            pl.col('Temp').str.extract(r'-\s*([a-zA-Z]+)', 1).alias('Status'),
        ]).with_columns([
            pl.when(pl.col('Temp').is_in(['0', None])).then(None).otherwise(pl.col(c)).alias(c)
            for c in ['Hours', 'Status']
        ])

    file_paths = glob.glob(f'{folder_path}/*.xlsx') + glob.glob(f'{folder_path}/*.csv')
    att_list, ot_list = [], []
    hc_sub = hc_db.select(['Date', 'OracleID', 'Email Id', 'Status']).with_columns(
        pl.col('OracleID').cast(pl.String)
    )

    for file in file_paths:
        is_ot  = 'OT Normal Month'        in file
        is_att = 'Attendance Normal Month' in file
        if not (is_ot or is_att):
            continue
        try:
            df_raw = pl.read_excel(file) if file.endswith('.xlsx') else \
                     pl.read_csv(file, encoding='utf-8', infer_schema_length=1000)
        except Exception:
            df_raw = pl.read_csv(file, encoding='ISO-8859-1', truncate_ragged_lines=True)

        base_cols = ['OU', 'employee_code', 'employee_name', 'Employee Type']
        if is_ot:
            base_cols.append('OT Type')
        prefix   = 'day_' if is_ot else ''
        day_cols = [f'{prefix}{i}' for i in range(1, 32)]
        avail    = [c for c in (base_cols + day_cols) if c in df_raw.columns]

        df = df_raw.select(avail)
        if df.height < 2:
            continue

        h_row     = df.row(0)
        new_names = get_unique_cols(df.columns, h_row, is_ot=is_ot)
        df = (
            df[2:]
            .rename({old: new for old, new in zip(df.columns, new_names)})
            .filter(pl.col('OU').str.contains('Vietnam'))
            .with_columns(pl.col('employee_code').cast(pl.String))
            .melt(id_vars=base_cols,
                  variable_name='Date',
                  value_name='ramco_marked' if is_att else 'Temp')
            .filter(pl.col('Date').str.contains(r'\d{2}-\w{3}-\d{4}'))
            .with_columns(pl.col('Date').str.strptime(pl.Date, '%d-%b-%Y', strict=False))
            .join(hc_sub, left_on=['Date', 'employee_code'],
                  right_on=['Date', 'OracleID'], how='left')
            .filter(pl.col('Status') == 'Active')
            .select(pl.all().exclude(['OU', 'Email Id', 'Status']))
        )
        (ot_list if is_ot else att_list).append(df.unique())

    RAMCO = (
        pl.concat(att_list).unique(subset=['employee_code', 'Date'])
        .rename({'employee_code': 'EID'})
        if att_list else pl.DataFrame()
    )

    if ot_list:
        ot_raw = extract_ot_details(pl.concat(ot_list))
        RAMCO_OT = (
            ot_raw
            .with_columns([
                pl.col('Hours').fill_null(0).alias('H_Num'),
                pl.col('employee_code').alias('EID'),
                pl.col('OT Type').str.contains('NSA').fill_null(False).alias('is_nsa'),
                pl.col('OT Type').str.contains('OT').fill_null(False).alias('is_ot'),
            ])
            .group_by(['EID', 'Date'])
            .agg([
                pl.col('OT Type').filter(pl.col('is_ot') & (pl.col('H_Num') > 0)).first()
                  .fill_null(
                      pl.col('OT Type').filter(pl.col('is_nsa') & pl.col('Status').is_not_null()).first()
                  )
                  .alias('ot_type'),
                
                pl.col('H_Num').filter(pl.col('is_ot')).sum().alias('hours'),
                
                pl.col('Status').filter(pl.col('is_nsa') & pl.col('Status').is_not_null()).last().alias('nsa_authorize'),
                pl.col('Status').filter(pl.col('is_ot') & pl.col('Status').is_not_null()).last().alias('ot_authorize'),
            ])
            .with_columns(
                pl.when(pl.col('hours') > 0)
                  .then(
                      pl.lit('(') + pl.col('hours').cast(pl.String) +
                      pl.lit(' hrs - ') + pl.col('ot_authorize').fill_null('Unknown') + pl.lit(')')
                  )
                  .otherwise(pl.lit(''))
                  .alias('ot_status')
            )
            .fill_null('')
            .filter(pl.col('hours') > 0)
        )
    else:
        RAMCO_OT = pl.DataFrame()

    return RAMCO, RAMCO_OT


RAMCO, RAMCO_OT = process_all_ramco_data(folder_paths['input_ramco_code'], HC_MASTER_DATABASE)
RAMCO.write_csv('ramco_remark.csv')
RAMCO_OT.write_csv('ramco_ot.csv')
print(RAMCO_OT)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21968\483673454.py:67: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  .melt(id_vars=base_cols,


shape: (1_882, 7)
┌───────────┬────────────┬─────────┬───────┬───────────────┬──────────────┬────────────────────────┐
│ EID       ┆ Date       ┆ ot_type ┆ hours ┆ nsa_authorize ┆ ot_authorize ┆ ot_status              │
│ ---       ┆ ---        ┆ ---     ┆ ---   ┆ ---           ┆ ---          ┆ ---                    │
│ str       ┆ date       ┆ str     ┆ f64   ┆ str           ┆ str          ┆ str                    │
╞═══════════╪════════════╪═════════╪═══════╪═══════════════╪══════════════╪════════════════════════╡
│ 103173195 ┆ 2026-04-30 ┆ OT3.0X  ┆ 8.0   ┆               ┆ Authorized   ┆ (8.0 hrs - Authorized) │
│ 103105143 ┆ 2026-01-21 ┆ OT2.1X  ┆ 2.5   ┆ Authorized    ┆ Authorized   ┆ (2.5 hrs - Authorized) │
│ 103169814 ┆ 2025-09-02 ┆ OT3.0X  ┆ 8.0   ┆               ┆ Authorized   ┆ (8.0 hrs - Authorized) │
│ 103082112 ┆ 2026-01-14 ┆ OT1.5X  ┆ 2.0   ┆               ┆ Authorized   ┆ (2.0 hrs - Authorized) │
│ 103092092 ┆ 2026-04-30 ┆ OT3.9X  ┆ 8.0   ┆               ┆ Authorized  

In [53]:
# ── RAMCO MONTHLY WORKING DAYS ────────────────────────────────────────────────

remark_col = 'ramco_marked' if 'ramco_marked' in RAMCO.columns else 'ramco_remark'
WORKING_SYMBOLS = ['PR', 'NM', 'PH', 'HO']

RAMCO_monthly = (
    RAMCO
    .with_columns([
        pl.col('EID').cast(pl.String).str.replace_all(r'\D', '').cast(pl.Int64, strict=False),
        pl.col('Date').cast(pl.Date),
        pl.when(pl.col(remark_col).is_in(WORKING_SYMBOLS)).then(1).otherwise(0).alias('working_flag'),
    ])
    .filter(pl.col('EID').is_not_null())
    .group_by([
        pl.col('Date').dt.truncate('1mo').alias('_MonthStart'),
        'EID', 'employee_name',
    ])
    .agg(pl.sum('working_flag').alias('count_working_days'))
    .with_columns(pl.col('_MonthStart').dt.strftime('%b-%y').alias('Month'))
    .select(['Month', 'EID', 'employee_name', 'count_working_days'])
    .sort(['Month', 'EID'])
)

RAMCO_monthly.write_excel('ramco_count_working_days.xlsx')
print(RAMCO_monthly.head())

shape: (5, 4)
┌────────┬───────────┬──────────────────────────┬────────────────────┐
│ Month  ┆ EID       ┆ employee_name            ┆ count_working_days │
│ ---    ┆ ---       ┆ ---                      ┆ ---                │
│ str    ┆ i64       ┆ str                      ┆ i32                │
╞════════╪═══════════╪══════════════════════════╪════════════════════╡
│ Apr-25 ┆ 101993774 ┆ HIEU THI TRAN            ┆ 22                 │
│ Apr-25 ┆ 102029874 ┆ THIEN THANH TOAN  TRUONG ┆ 20                 │
│ Apr-25 ┆ 102173363 ┆ THI NGOC BICH  TRAN      ┆ 23                 │
│ Apr-25 ┆ 102173384 ┆ HUU VINH  MAI            ┆ 22                 │
│ Apr-25 ┆ 102211178 ┆ THI QUYNH TRANG  NGUYEN  ┆ 22                 │
└────────┴───────────┴──────────────────────────┴────────────────────┘


In [ ]:
# ── PARSE DATETIME COLUMNS (vectorised) ───────────────────────────────────────

AGENT_ACTIVITY_INPUT = AGENT_ACTIVITY_INPUT.with_columns([
    pl.Series('Start Time', parse_datetime_col(AGENT_ACTIVITY_INPUT['Start Time'])),
    pl.Series('End Time',   parse_datetime_col(AGENT_ACTIVITY_INPUT['End Time'])),
])

AGENT_ACTIVITY_INPUT_FOR_NS = AGENT_ACTIVITY_INPUT.with_columns([
    pl.col('Start Time').str.strptime(pl.Datetime, '%Y-%m-%d %H:%M:%S', strict=False),
    pl.col('End Time').str.strptime(pl.Datetime, '%Y-%m-%d %H:%M:%S', strict=False),
]).with_columns([
    pl.col('Start Time').dt.date().alias('Start Date'),
    pl.col('End Time').dt.date().alias('End Date'),
    (pl.col('Start Time').dt.date() + pl.duration(days=1)).alias('Next Date'),
    (pl.col('Start Time').dt.date() - pl.duration(days=1)).alias('Previous Date'),
])

In [ ]:
# ── NIGHT SHIFT MERGE & UPDATE ────────────────────────────────────────────────

# 1. Khởi tạo Previous Night Shift
Prev_NS = (
    Night_Shift_1
    .with_columns((pl.col('Date') + pl.duration(days=1)).alias('Date'))
    .rename({'Night_Shift': 'Previous_Night_Shift'})
)

# 2. Lấy thêm thông tin ca của NGÀY HÔM NAY (Today) để tạo "Lực Hút"
IEX_Today = IEX[['Date', 'Email Id', 'Datetime_Fluctuate_Start_Shift']].unique()

ACTIVITY_MERGED_NIGHT_SHIFT = (
    AGENT_ACTIVITY_INPUT_FOR_NS
    .join(
        Night_Shift_1,
        left_on=['Start Date', 'Agent Email'],
        right_on=['Date', 'Email Id'],
        how='left',
    )
    .join(
        Prev_NS,
        left_on=['Start Date', 'Agent Email'],
        right_on=['Date', 'Email Id'],
        how='left',
    )
    .join(
        IEX_Today,
        left_on=['Start Date', 'Agent Email'],
        right_on=['Date', 'Email Id'],
        how='left',
    )
    .with_columns([
        pl.col('Night_Shift').cast(pl.Float64).fill_null(0.0),
        pl.col('Previous_Night_Shift').cast(pl.Float64).fill_null(0.0),
    ])
    .rename({'Start Date': 'Date'})
)

# 3. Phân loại và Convert Ngày cực chuẩn bằng Lực Hút Của Ca Làm Việc (Shift Gravity)
def update_night_shift(df: pl.DataFrame) -> pl.DataFrame:

    hr = pl.col('Start Time').dt.hour()

    hours_to_shift = (
        (pl.col('Datetime_Fluctuate_Start_Shift') - pl.col('Start Time'))
        .dt.total_seconds() / 3600
    )

    # ── Case 2: OT sáng cùng ngày với ca đêm hôm nay ─────────────────────────
    # Previous_Night_Shift = 0, Night_Shift = 1, activity trước shift start buổi tối
    # → Preshift OT: giữ nguyên ngày hôm nay, KHÔNG convert về hôm trước
    is_preshift_ot_today = (
        (pl.col('Previous_Night_Shift') == 0.0) &
        (pl.col('Night_Shift') == 1.0) &
        pl.col('Datetime_Fluctuate_Start_Shift').is_not_null() &
        (hours_to_shift > 1)   # activity xảy ra trước shift start > 1h
    )

    # ── Case 1 & 3: Convert về ngày hôm trước ─────────────────────────────────
    # Điều kiện chung: Previous_Night_Shift = 1 AND activity trước 15:00
    # Case 1: Night_Shift = 1 hôm nay → vẫn convert (ca liên tục đêm → sáng)
    # Case 3: Night_Shift = 0 hôm nay → convert phần < 15:00, giữ phần >= 15:00
    is_ext = (
        (~is_preshift_ot_today) &
        (pl.col('Previous_Night_Shift') == 1.0) &
        (hr < 15)
    )

    df = df.with_columns(
        pl.when(is_ext)
          .then(pl.col('Date') - pl.duration(days=1))
          .otherwise(pl.col('Date'))
          .alias('Date_Converted')
    )

    df = df.with_columns(
        pl.when(is_ext).then(1.0)
          .otherwise(pl.col('Night_Shift'))
          .alias('Night_Shift')
    )

    # Drop helper columns only if they exist
    cols_to_drop = [c for c in ['Datetime_Fluctuate_Start_Shift', 'Prev_Fluctuate_Start_Shift'] if c in df.columns]
    return df.drop(cols_to_drop)


ACTIVITY_TOTAL_CONVERTED = update_night_shift(ACTIVITY_MERGED_NIGHT_SHIFT)

In [14]:
# ── SPLIT IEX vs NON-IEX & REBUILD df_result ──────────────────────────────────

Merged_With_IEX = ACTIVITY_TOTAL_CONVERTED.join(
    IEX[['Date', 'Email Id', 'First Shift']],
    left_on=['Date_Converted', 'Agent Email'],
    right_on=['Date', 'Email Id'],
    how='left',
)

Agent_Activity_Include_IEX = Merged_With_IEX.filter(~pl.col('First Shift').is_null())
Agent_Not_IEX              = Merged_With_IEX.filter( pl.col('First Shift').is_null())

def add_tracker_time(df, agent_col, date_col, time_col, new_col, agg_func, threshold=15, direction='>'):
    filt = (
        df.filter(pl.col(time_col).dt.hour() > threshold)
        if direction == '>'
        else df.filter(pl.col(time_col).dt.hour() < threshold)
    )
    if filt.is_empty():
        return df.with_columns(pl.lit(None).cast(pl.Datetime).alias(new_col))
    agg = filt.group_by([agent_col, date_col]).agg(
        getattr(pl.col(time_col), agg_func)().alias(new_col)
    )
    return df.join(agg, on=[agent_col, date_col], how='left')

# Khởi tạo Tracker cho nhóm Non-IEX
AGENT_ACTIVITY_TRACKER = add_tracker_time(Agent_Not_IEX, 'Agent Email', 'Date', 'Start Time', 'Min Date < 3pm', 'min', 15, '<')
AGENT_ACTIVITY_TRACKER = add_tracker_time(AGENT_ACTIVITY_TRACKER, 'Agent Email', 'Date', 'Start Time', 'Max Date < 3pm', 'max', 15, '<')

tracker_ns = (
    AGENT_ACTIVITY_TRACKER
    .group_by(['Agent Email', 'Date'])
    .agg([
        pl.col('Start Time').filter(pl.col('Start Time').dt.hour() < 4).min().alias('Min Date+1 < 4am'),
        pl.col('Start Time').filter(pl.col('Start Time').dt.hour() < 15).max().alias('Max Date+1 < 3pm'),
    ])
    .with_columns(
        pl.when(pl.col('Min Date+1 < 4am').is_not_null() & pl.col('Max Date+1 < 3pm').is_not_null())
          .then(1.0).otherwise(0.0).alias('Night_Shift_New')
    )
    .sort('Date')
    .with_columns(
        pl.col('Night_Shift_New').shift(1).over('Agent Email').alias('Prev_NS_New')
    )
)

Agent_Activity_Exclude_IEX = (
    Agent_Not_IEX
    .join(
        tracker_ns[['Date', 'Agent Email', 'Night_Shift_New', 'Prev_NS_New']],
        on=['Date', 'Agent Email'],
        how='left',
    )
    .with_columns([
        pl.when(pl.col('Night_Shift_New').is_not_null() & (pl.col('Night_Shift_New') == 1))
          .then(pl.col('Night_Shift_New')).otherwise(pl.col('Night_Shift')).fill_null(0.0).alias('Night_Shift'),
        pl.when(pl.col('Prev_NS_New').is_not_null() & (pl.col('Prev_NS_New') == 1))
          .then(pl.col('Prev_NS_New')).otherwise(pl.col('Previous_Night_Shift')).fill_null(0.0).alias('Previous_Night_Shift'),
    ])
    .drop(['Night_Shift_New', 'Prev_NS_New'])
)

# [QUAN TRỌNG]: Bổ sung cột Datetime_Fluctuate_Start_Shift rỗng (Null) cho nhóm Non-IEX 
# để tương thích với thuật toán "Shift Gravity" mới ở Cell 17
Agent_Activity_Exclude_IEX = Agent_Activity_Exclude_IEX.with_columns(
    pl.lit(None).cast(pl.Datetime).alias('Datetime_Fluctuate_Start_Shift')
)

# Chạy lại convert ngày cho nhóm Exclude (Non-IEX)
Agent_Activity_Exclude_IEX = update_night_shift(Agent_Activity_Exclude_IEX.drop('Date_Converted'))

# Ép lại kiểu Float cho an toàn trước khi nối
Agent_Activity_Include_IEX = Agent_Activity_Include_IEX.with_columns([
    pl.col('Night_Shift').cast(pl.Float64),
    pl.col('Previous_Night_Shift').cast(pl.Float64)
])
Agent_Activity_Exclude_IEX = Agent_Activity_Exclude_IEX.with_columns([
    pl.col('Night_Shift').cast(pl.Float64),
    pl.col('Previous_Night_Shift').cast(pl.Float64)
])

# Nối bảng
df_result = pl.concat([Agent_Activity_Include_IEX, Agent_Activity_Exclude_IEX], how='diagonal')

In [15]:
# ── CLEAN REFERENCE TABLES & MAIN JOIN ───────────────────────────────────────

HC_CLEAN = (
    HC_MASTER_DATABASE
    .select(['Year', 'Month', 'Week_Monday', 'Date', 'OracleID', 'IEX ID',
             'Employee Name', 'Email Id', 'LOB', 'LOB_2', 'LOB_3', 'Alias',
             'Supervisor Name', 'Wave', 'Detail Status', 'Status'])
    .unique()
    .rename({'Date': 'Date_Converted'})
    .group_by(['Date_Converted', 'Email Id'])
    .first()
)

IEX_COLS = [
    'Year', 'Month', 'Week_Monday', 'Date', 'Employee Name', 'Email Id', 'OracleID', 'IEX_ID',
    'LOB', 'LOB_2', 'LOB_3', 'Alias', 'Supervisor Name', 'Wave', 'Detail Status', 'Status',
    'Night_Shift', 'Target', 'Datetime_Fluctuate_Start_Shift', 'Datetime_Fluctuate_End_Shift',
    'Fluctuate Shift', 'Datetime_First_Start_Shift', 'Datetime_First_End_Shift', 'First Shift',
    'Time_Of_Day', 'Open Time', 'Extra Time', 'Break Time', 'Lunch Time', 'Training',
    'NCNS', 'AL', 'Unplanned', 'Planned', 'Roster Presented', 'Roster Scheduled',
    'Shift Tracking', 'OT Type', 'Combined OT Range', 'OT PreShift', 'OT PostShift',
]

COMPACT_IEX_CLEAN = (
    IEX.select([c for c in IEX_COLS if c in IEX.columns])
    .unique()
    .rename({'Date': 'Date_Converted', 'IEX_ID': 'IEX ID'})
    .group_by(['Date_Converted', 'Email Id'])
    .first()
)

min_date = df_result.select(pl.col('Date_Converted').min()).item()
COMPACT_IEX_CLEAN = COMPACT_IEX_CLEAN.filter(pl.col('Date_Converted') >= min_date)

final_df = (
    df_result
    .join(HC_CLEAN,          left_on=['Date_Converted', 'Agent Email'], right_on=['Date_Converted', 'Email Id'], how='left')
    .join(COMPACT_IEX_CLEAN, left_on=['Date_Converted', 'Agent Email'], right_on=['Date_Converted', 'Email Id'], how='left', suffix='_IEX')
)

ACTIVITY_DATE_CONVERTED = (
    final_df
    .with_columns([
        pl.col('Year').cast(pl.String),
        pl.col('OracleID').cast(pl.String),
        pl.col('IEX ID').cast(pl.String),
    ])
    .rename({'Agent Email': 'Email Id'})
    .sort('Start Time')
)

print(ACTIVITY_DATE_CONVERTED
      .with_columns(pl.col('Agent State').str.strip_chars())
      ['Agent State'].unique().sort().to_list())

[None, 'After Call Work', 'Available Chat', 'Available Phone', 'Break', 'Coaching', 'End of Shift', 'Login', 'Lunch', 'Not Ready', 'Offline', 'Outbound Call', 'Phone Talking', 'Restroom', 'System Issue', 'Team Meeting', 'Training']


In [16]:
# ── LODGING PRODUCTIVITY SUMMARY ──────────────────────────────────────────────

BREAK_KW     = ['Lunch', 'Break']
ACT_PROD_KW  = ['Coaching', 'Team Meeting', 'Training']
NON_PROD_KW  = ['End of Shift', 'Login', 'Not Ready', 'Offline', 'Restroom', 'System Issue']
AVAIL_STATES = ['Available Chat', 'Outbound Call', 'Available Phone']

agent_act = ACTIVITY_DATE_CONVERTED.filter(pl.col('LOB') == 'Lodging')

final_report = (
    agent_act
    .with_columns(
        pl.when(
            (pl.col('Productive Aux Flag (Yes / No)') == 'Yes') &
            (pl.col('Number of Active Contacts').cast(pl.Int64, strict=False) > 0)
        ).then(pl.lit('THT'))
        .when(
            (pl.col('Number of Active Contacts').cast(pl.Int64, strict=False) == 0) &
            pl.col('Agent State').is_in(AVAIL_STATES)
        ).then(pl.lit('Avail Time'))
        .when(pl.col('Agent State').str.contains('|'.join(BREAK_KW))).then(pl.lit('Total Break'))
        .when(pl.col('Agent State').str.contains('|'.join(NON_PROD_KW))).then(pl.lit('Non Productive Aux'))
        .when(pl.col('Agent State').str.contains('|'.join(ACT_PROD_KW))).then(pl.lit('Other Productive Aux'))
        .alias('Category')
    )
    .group_by('Date_Converted')
    .agg([
        (pl.col('Total Time in seconds').sum() / 3600).alias('Staff Hour (In Hours)'),
        (pl.col('Total Time in seconds').filter(pl.col('Category') == 'Total Break').sum() / 3600).alias('Total Break (In Hours)'),
        (pl.col('Total Time in seconds').filter(pl.col('Category') == 'THT').sum() / 3600).alias('THT(In Hours)'),
        (pl.col('Total Time in seconds').filter(pl.col('Category') == 'Avail Time').sum() / 3600).alias('Avail Time(In Hours)'),
        (pl.col('Total Time in seconds').filter(pl.col('Category') == 'Other Productive Aux').sum() / 3600).alias('Other Productive Aux(In Hours)'),
        (pl.col('Total Time in seconds').filter(pl.col('Category') == 'Non Productive Aux').sum() / 3600).alias('Non Productive Aux (In Hours)'),
        pl.col('Email Id').n_unique().alias('Present for Calculation'),
    ])
    .sort('Date_Converted')
)

print(final_report)
final_report.write_excel('sea_productive.xlsx')

shape: (120, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ Date_Conve ┆ Staff Hour ┆ Total      ┆ THT(In    ┆ Avail     ┆ Other Pro ┆ Non Produ ┆ Present   │
│ rted       ┆ (In Hours) ┆ Break (In  ┆ Hours)    ┆ Time(In   ┆ ductive   ┆ ctive Aux ┆ for Calcu │
│ ---        ┆ ---        ┆ Hours)     ┆ ---       ┆ Hours)    ┆ Aux(In    ┆ (In       ┆ lation    │
│ date       ┆ f64        ┆ ---        ┆ f64       ┆ ---       ┆ Hours)    ┆ Hours)    ┆ ---       │
│            ┆            ┆ f64        ┆           ┆ f64       ┆ ---       ┆ ---       ┆ u32       │
│            ┆            ┆            ┆           ┆           ┆ f64       ┆ f64       ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2026-01-05 ┆ 870.359225 ┆ 109.324353 ┆ 485.61549 ┆ 198.67493 ┆ 6.482992  ┆ 70.26145  ┆ 76        │
│            ┆            ┆            ┆ 7         ┆ 3         ┆           

In [17]:
# ── GENERATE RTA REPORT ───────────────────────────────────────────────────────

def generate_rta_report(
    df_activity: pl.DataFrame,
    df_pull_out: pl.DataFrame,
    df_ramco: pl.DataFrame,
    df_ramco_ot: pl.DataFrame,
) -> pl.DataFrame:

    GROUP_COLS = [
        'Year', 'Month', 'Week_Monday', 'Date_Converted', 'Employee Name', 'Email Id',
        'OracleID', 'IEX ID', 'Target', 'Datetime_Fluctuate_Start_Shift',
        'Datetime_Fluctuate_End_Shift', 'Fluctuate Shift', 'Datetime_First_Start_Shift',
        'Datetime_First_End_Shift', 'First Shift', 'Alias', 'LOB', 'LOB_2', 'LOB_3',
        'Supervisor Name', 'Wave', 'Detail Status', 'Status', 'Time_Of_Day',
        'Open Time', 'Extra Time', 'Break Time', 'Lunch Time', 'Training',
        'NCNS', 'AL', 'Unplanned', 'Planned', 'Roster Presented', 'Roster Scheduled',
        'Shift Tracking', 'OT Type', 'Combined OT Range', 'OT PreShift', 'OT PostShift',
    ]
    valid_group = [c for c in GROUP_COLS if c in df_activity.columns]

    def _sw(cond):
        return pl.when(cond).then(pl.col('Total Time in seconds')).sum().cast(pl.Float64)

    # 1. Aggregation
    df_agg = df_activity.group_by(valid_group).agg([
        pl.col('Start Time').min().alias('start'),
        pl.col('End Time').max().alias('end'),
        pl.col('Total Time in seconds').sum().cast(pl.Float64).alias('duration'),
        pl.col('Night_Shift').max().alias('Night_Shift'),
        pl.when(pl.col('Number of Active Contacts').is_not_null())
          .then(pl.col('Total Time in seconds')).sum().cast(pl.Float64).alias('total_time_chat_handle'),
        pl.when(pl.col('Productive Aux Flag (Yes / No)') == 'Yes')
          .then(pl.col('Total Time in seconds')).sum().cast(pl.Float64).alias('sum_productive'),
        _sw((pl.col('Agent State').is_in(['Break-IDLE', 'Break'])) & (pl.col('Productive Aux Flag (Yes / No)') == 'No')).alias('break'),
        _sw((pl.col('Agent State').is_in(['Lunch-IDLE', 'Lunch'])) & (pl.col('Productive Aux Flag (Yes / No)') == 'No')).alias('lunch'),
        _sw((pl.col('Agent State').is_in(['Coaching-IDLE', 'Coaching'])) & (pl.col('Productive Aux Flag (Yes / No)') == 'No')).alias('coaching-idle'),
        _sw((pl.col('Agent State').is_in(['Training-IDLE', 'Training'])) & (pl.col('Productive Aux Flag (Yes / No)') == 'No')).alias('training-idle'),
        _sw((pl.col('Agent State') == 'Outbound Call-IDLE') & (pl.col('Productive Aux Flag (Yes / No)') == 'No')).alias('outbound-idle'),
        pl.when(
            (pl.col('Agent State') == 'Break-IDLE') &
            (pl.col('Total Time in seconds') > 60) &
            (pl.col('Productive Aux Flag (Yes / No)') == 'No')
        ).then(pl.col('Total Time in seconds')).count().cast(pl.Float64).alias('break_count'),
        _sw(
            (pl.col('Productive Aux Flag (Yes / No)') == 'No') &
            ~pl.col('Agent State').is_in([
                'Break-IDLE', 'Break', 'Lunch-IDLE', 'Lunch',
                'Coaching-IDLE', 'Coaching', 'Training-IDLE', 'Training', 'Outbound Call-IDLE',
            ])
        ).alias('other_status'),
    ])

    # 2. Derived metrics
    ds = pl.col('Detail Status')
    df_calc = (
        df_agg
        .with_columns([
            pl.when(pl.col('break').is_null()).then(0)
              .when(ds.str.contains('Production')).then(pl.max_horizontal(pl.lit(0), pl.col('break') - 1800))
              .when(ds.str.contains('Nesting')).then(pl.max_horizontal(pl.lit(0), pl.col('break') - 900))
              .otherwise(0).alias('over_break'),
            pl.when(pl.col('lunch').is_null()).then(0)
              .otherwise(pl.max_horizontal(pl.lit(0), pl.col('lunch') - 3600)).alias('over_lunch'),
            pl.when((pl.col('break_count') == 0) | pl.col('break_count').is_null()).then(0)
              .when(ds.str.contains('Production')).then(pl.max_horizontal(pl.lit(0), pl.col('break_count') - 2))
              .when(ds.str.contains('Nesting')).then(pl.max_horizontal(pl.lit(0), pl.col('break_count') - 1))
              .otherwise(0).alias('exceed_break'),
            pl.when(pl.col('duration') > 0)
              .then(
                  pl.when(ds.str.contains('Nesting'))
                    .then(pl.when(pl.col('duration') >= 10800).then(1).when(pl.col('duration') >= 7200).then(0.5).otherwise(0))
                    .otherwise(pl.when(pl.col('duration') >= 25200).then(1).when(pl.col('duration') >= 10800).then(0.5).otherwise(0))
              )
              .otherwise(0).alias('hc_actual'),
        ])
        .with_columns([
            pl.when(pl.col('First Shift').str.contains('-') & (pl.col('Time_Of_Day') > 0) & (pl.col('Time_Of_Day') < 18000))
              .then(0.5)
            .when(pl.col('First Shift').str.contains('-') & (pl.col('Time_Of_Day') >= 18000))
              .then(1)
            .when(pl.col('First Shift') == 'AL').then(1)
            .otherwise(0).cast(pl.Float64).alias('hc_schedule'),
            pl.when((pl.col('start') - pl.col('Datetime_Fluctuate_Start_Shift')).dt.total_seconds() > 180)
              .then((pl.col('start') - pl.col('Datetime_Fluctuate_Start_Shift')).dt.total_seconds())
              .otherwise(0).cast(pl.Float64).alias('time_late'),
            pl.when((pl.col('Datetime_Fluctuate_End_Shift') - pl.col('end')).dt.total_seconds() > 0)
              .then((pl.col('Datetime_Fluctuate_End_Shift') - pl.col('end')).dt.total_seconds())
              .otherwise(0).cast(pl.Float64).alias('time_leave'),
        ])
        .with_columns([
            pl.when(pl.col('start').is_not_null() & (pl.col('time_late') > 0))
              .then(1).otherwise(0).cast(pl.Float64).alias('lateness'),
            (pl.col('Target') - pl.col('time_late') - pl.col('over_break') - pl.col('over_lunch'))
              .alias('adherence_time'),
        ])
    )

    # 3. Joins (Pull Out + Ramco + Ramco OT)
    df_merged = (
        df_calc
        .join(df_pull_out.select(['Pull-out Date', 'Emp ID', 'total_time_pull_out']),
              left_on=['Date_Converted', 'OracleID'], right_on=['Pull-out Date', 'Emp ID'], how='left')
        .join(df_ramco.select(['EID', 'Date', 'ramco_marked']),
              left_on=['Date_Converted', 'OracleID'], right_on=['Date', 'EID'], how='left')
        .join(df_ramco_ot.select(['EID', 'Date', 'nsa_authorize', 'ot_authorize', 'hours', 'ot_status', 'ot_type']),
              left_on=['Date_Converted', 'OracleID'], right_on=['Date', 'EID'], how='left')
    )

    # 4. Seconds -> hours (single pass)
    SEC_TO_HR = [
        'Target', 'Time_Of_Day', 'Open Time', 'Extra Time', 'Break Time', 'Lunch Time',
        'Training', 'NCNS', 'AL', 'total_time_chat_handle', 'sum_productive', 'break',
        'lunch', 'outbound-idle', 'coaching-idle', 'training-idle', 'over_break', 'over_lunch',
        'duration', 'time_late', 'time_leave', 'adherence_time', 'total_time_pull_out', 'other_status',
    ]
    convert_cols = [c for c in SEC_TO_HR if c in df_merged.columns]
    df_merged = df_merged.with_columns([
        (pl.col(c).fill_null(0).cast(pl.Float64) / 3600).alias(c) for c in convert_cols
    ])

    # 5. Aging buckets
    def _bucket(days_col):
        return (
            pl.when(days_col.is_between(0, 2)).then(pl.lit('00-02 Days'))
            .when(days_col.is_between(3, 5)).then(pl.lit('03-05 Days'))
            .when(days_col.is_between(6, 10)).then(pl.lit('06-10 Days'))
            .when(days_col.is_between(11, 30)).then(pl.lit('>10 Days'))
            .when(days_col > 30).then(pl.lit('>30 Days Unapproved'))
            .otherwise(None)
        )

    today_date = datetime.now().date()
    return (
        df_merged
        .with_columns(
            (pl.lit(today_date).cast(pl.Date) - pl.col('Date_Converted'))
            .dt.total_days().alias('days_count')
        )
        .with_columns([
            pl.when(pl.col('ramco_marked') == 'NM')
              .then(_bucket(pl.col('days_count'))).otherwise(None).alias('nm_group'),
            pl.when(pl.col('ot_type').str.contains('NSA') & (pl.col('nsa_authorize') == 'Pending'))
              .then(_bucket(pl.col('days_count'))).otherwise(None).alias('nsa_group'),
            pl.when(pl.col('ot_type').str.contains('OT') & (pl.col('ot_authorize') == 'Pending'))
              .then(_bucket(pl.col('days_count'))).otherwise(None).alias('ot_group'),
        ])
        .sort(['Date_Converted', 'start'], descending=[False, True])
        .unique()
    )


RTA_REPORT = generate_rta_report(ACTIVITY_DATE_CONVERTED, PULL_OUT, RAMCO, RAMCO_OT)

In [18]:
# ── POST-PROCESS RTA_REPORT ───────────────────────────────────────────────────

DESIRED_ORDER = [
    'Year', 'Month', 'Week_Monday', 'Date_Converted', 'Employee Name', 'Email Id',
    'OracleID', 'IEX ID', 'Target', 'Datetime_Fluctuate_Start_Shift',
    'Datetime_Fluctuate_End_Shift', 'Fluctuate Shift', 'Datetime_First_Start_Shift',
    'Datetime_First_End_Shift', 'First Shift', 'Alias', 'LOB', 'LOB_2', 'LOB_3',
    'Supervisor Name', 'Wave', 'Detail Status', 'Status', 'Time_Of_Day', 'Open Time',
    'Extra Time', 'Break Time', 'Lunch Time', 'Training', 'NCNS', 'AL', 'Unplanned',
    'Planned', 'Roster Presented', 'Roster Scheduled', 'Night_Shift', 'Shift Tracking',
    'OT Type', 'Combined OT Range', 'OT PreShift', 'OT PostShift', 'start', 'end',
    'total_time_chat_handle', 'sum_productive', 'break', 'lunch', 'coaching-idle',
    'training-idle', 'outbound-idle', 'break_count', 'other_status', 'over_break',
    'over_lunch', 'exceed_break', 'duration', 'hc_actual', 'hc_schedule', 'time_late',
    'time_leave', 'lateness', 'adherence_time', 'total_time_pull_out', 'ramco_marked',
    'nsa_authorize', 'ot_authorize', 'hours', 'ot_status', 'ot_type', 'days_count',
    'nm_group', 'nsa_group', 'ot_group',
]

# Merge COMPACT_IEX_CLEAN + RTA_REPORT, keep RTA row on key collision
RTA_REPORT = (
    pl.concat(
        [COMPACT_IEX_CLEAN.with_columns(pl.lit('COMPACT').alias('_src')),
         RTA_REPORT.with_columns(pl.lit('RTA').alias('_src'))],
        how='diagonal',
    )
    .unique(subset=['Date_Converted', 'Email Id'], keep='last', maintain_order=True)
    .drop('_src')
)

# Normalise IEX time cols that may still be in hours
TIME_NORM = ['Time_Of_Day', 'Open Time', 'Extra Time', 'Break Time',
             'Lunch Time', 'Training', 'NCNS', 'AL', 'Target']
RTA_REPORT = RTA_REPORT.with_columns([
    pl.when(pl.col(c).cast(pl.Float64) < 0).then(0.0)
      .when(pl.col(c).cast(pl.Float64) > 100).then(pl.col(c).cast(pl.Float64) / 3600.0)
      .otherwise(pl.col(c).cast(pl.Float64))
      .alias(c)
    for c in TIME_NORM if c in RTA_REPORT.columns
])

# Backfill ramco_marked where null
RTA_REPORT = (
    RTA_REPORT
    .join(
        RAMCO.select(['EID', 'Date', 'ramco_marked']).rename({'ramco_marked': '_rm_bk'}),
        left_on=['Date_Converted', 'OracleID'], right_on=['Date', 'EID'], how='left',
    )
    .with_columns(
        pl.when(pl.col('ramco_marked').is_null() | (pl.col('ramco_marked') == ''))
          .then(pl.col('_rm_bk')).otherwise(pl.col('ramco_marked')).alias('ramco_marked')
    )
    .drop('_rm_bk')
    .select([c for c in DESIRED_ORDER if c in RTA_REPORT.columns])
)

# ── CLEAN DATETIME FORMATTING ─────────────────────────────────────
DT_COLS_TO_CLEAN = [
    'Datetime_Fluctuate_Start_Shift', 'Datetime_Fluctuate_End_Shift',
    'Datetime_First_Start_Shift', 'Datetime_First_End_Shift',
    'start', 'end'
]

RTA_REPORT = RTA_REPORT.with_columns([
    pl.when(pl.col(c).is_not_null())
      .then(pl.col(c).dt.strftime('%Y-%m-%d %H:%M:%S'))
      .otherwise(pl.col(c))
      .alias(c)
    for c in DT_COLS_TO_CLEAN if c in RTA_REPORT.columns
])

In [19]:
# ── GENERATE 30-MIN INTERVALS (RTA_EXTEND) ────────────────────────────────────

VNT_TZ = 'Asia/Ho_Chi_Minh'
PST_TZ = 'America/Los_Angeles'

def _to_pst(col_expr):
    return (
        col_expr
        .dt.replace_time_zone(VNT_TZ, ambiguous='earliest')
        .dt.convert_time_zone(PST_TZ)
        .dt.replace_time_zone(None)
    )

LOB_MAP = pl.DataFrame({
    'Agent Queue Group Name': ['Chat_OD_EN_Dual_GDS', 'Chat_OD_EN_Car_Activity',
                               'Chat_OD_EN_Lodging', 'Chat_OD_EN_Amadeus'],
    'LOB': ['Non_Lodging', 'Lodging', 'Lodging', 'Non_Lodging'],
})

RTA_EXTEND = (
    ACTIVITY_DATE_CONVERTED
    .with_columns([
        pl.col('Start Time').cast(pl.Datetime),
        pl.col('End Time').cast(pl.Datetime),
    ])
    .with_columns(
        pl.when(pl.col('End Time') < pl.col('Start Time'))
          .then(pl.col('End Time') + pl.duration(days=1))
          .otherwise(pl.col('End Time')).alias('End Time')
    )
    .with_columns(
        pl.datetime_ranges(
            start=pl.col('Start Time').dt.truncate('30m'),
            end=pl.col('End Time'),
            interval='30m',
            closed='left',
        ).alias('_ck')
    )
    .explode('_ck')
    .with_columns([
        pl.max_horizontal([pl.col('Start Time'), pl.col('_ck')]).alias('Datetime_Start_Time'),
        pl.min_horizontal([pl.col('End Time'), pl.col('_ck') + pl.duration(minutes=30)]).alias('Datetime_End_Time'),
        _to_pst(pl.col('_ck')).alias('_ck_pst'),
    ])
    .with_columns([
        _to_pst(pl.col('Datetime_Start_Time')).alias('PST_Start_Time'),
        _to_pst(pl.col('Datetime_End_Time')).alias('PST_End_Time'),
    ])
    .filter(pl.col('Datetime_End_Time') > pl.col('Datetime_Start_Time'))
    .with_columns([
        ((pl.col('Datetime_End_Time') - pl.col('Datetime_Start_Time')).dt.total_seconds() / 3600).alias('Duration'),
        pl.col('_ck').dt.strftime('%Y-%m-%d %H:%M:%S').alias('VNT_Intervals'),
        pl.col('_ck_pst').dt.strftime('%Y-%m-%d %H:%M:%S').alias('PST_Intervals'),
        (pl.col('_ck').dt.strftime('%H:%M') + '-' + (pl.col('_ck') + pl.duration(minutes=30)).dt.strftime('%H:%M')).alias('VNT_Interval_Range'),
        (pl.col('_ck_pst').dt.strftime('%H:%M') + '-' + (pl.col('_ck_pst') + pl.duration(minutes=30)).dt.strftime('%H:%M')).alias('PST_Interval_Range'),
        pl.col('Datetime_Start_Time').dt.strftime('%Y-%m-%d %H:%M:%S').alias('Datetime_Start_Time'),
        pl.col('Datetime_End_Time').dt.strftime('%Y-%m-%d %H:%M:%S').alias('Datetime_End_Time'),
    ])
    .with_columns(pl.col('Agent Queue Group Name').cast(pl.String).str.strip_chars())
    .join(LOB_MAP, on='Agent Queue Group Name', how='left')
    .with_columns(pl.col('LOB').fill_null('Unknown'))
    .drop(['_ck', '_ck_pst'])
)

TARGET_COLS = [
    'Month', 'Week_Monday', 'Date_Converted', 'Employee Name', 'Email Id', 'OracleID', 'IEX ID',
    'Agent Queue Group Name', 'Productive Aux Flag (Yes / No)', 'Agent State',
    'Datetime_Start_Time', 'Datetime_End_Time', 'Target', 'Duration', 'LOB',
    'VNT_Intervals', 'PST_Intervals', 'VNT_Interval_Range', 'PST_Interval_Range',
]
RTA_EXTEND = RTA_EXTEND.select([c for c in TARGET_COLS if c in RTA_EXTEND.columns]).unique()

In [20]:
# ── IEX INTERVALS -> ADHERENCE ────────────────────────────────────────────────

IEX_INTERVALS_INPUT = input_data(folder_paths['input_iex_intervals']).unique()

IEX_INTERVALS_PROCESSED = (
    IEX_INTERVALS_INPUT
    .with_columns([
        pl.col('PST_Intervals', 'VNT_Intervals').str.to_datetime(strict=False),
        pl.col('Duration').cast(pl.Float64),
    ])
    .group_by(['PST_Intervals', 'VNT_Intervals', 'Email Id'])
    .agg([
        pl.col('Duration').filter(pl.col('Scheduled Activity').is_in(['Open Time', 'Extra Hours'])).sum().alias('Scheduled_Productive'),
        pl.col('Duration').filter(~pl.col('Scheduled Activity').is_in(['Open Time', 'Extra Hours'])).sum().alias('Scheduled_Unproductive'),
        pl.col('Duration').filter(pl.col('Scheduled Activity').str.contains('Lunch')).sum().alias('Scheduled_Lunch'),
        pl.col('Duration').filter(pl.col('Scheduled Activity').str.contains('Break')).sum().alias('Scheduled_Break'),
        pl.col('Duration').filter(pl.col('Scheduled Activity').str.contains('Training|Coaching|Meeting')).sum().alias('Scheduled_Training'),
    ])
    .with_columns([
        pl.col('Scheduled_Productive').clip(upper_bound=0.5),
        pl.col('VNT_Intervals').dt.date().alias('VNT_Date'),
    ])
)

RTA_EXTEND_SUMMARY = (
    RTA_EXTEND
    .with_columns(pl.col('PST_Intervals').str.to_datetime(strict=False))
    .group_by(['PST_Intervals', 'Email Id'])
    .agg([
        pl.col('Duration').filter((pl.col('Productive Aux Flag (Yes / No)') == 'Yes') & (pl.col('Agent State') != 'Offline')).sum().alias('Productive'),
        pl.col('Duration').filter(pl.col('Productive Aux Flag (Yes / No)') == 'No').sum().alias('Non-Productive'),
        pl.col('Duration').filter((pl.col('Productive Aux Flag (Yes / No)') == 'No') & pl.col('Agent State').is_in(['Lunch', 'Break'])).sum().alias('Lunch/Break'),
        pl.col('Duration').filter((pl.col('Productive Aux Flag (Yes / No)') == 'No') & pl.col('Agent State').str.contains('Training|Coaching|Meeting')).sum().alias('Training'),
    ])
)

Adherence = (
    IEX_INTERVALS_PROCESSED
    .join(RTA_EXTEND_SUMMARY, on=['PST_Intervals', 'Email Id'], how='left')
    .with_columns([
        pl.col('Scheduled_Productive').fill_null(0),
        pl.col('Productive').fill_null(0),
        pl.col('PST_Intervals').dt.date().alias('PST_Date'),
    ])
    .with_columns(
        pl.min_horizontal('Productive', 'Scheduled_Productive').alias('Final_Productive')
    )
)

res = Path(folder_paths['resources'])
Adherence.write_parquet(res / 'Adherence.parquet')
Adherence.write_excel(res / 'Adherence.xlsx')

In [21]:
# ── EXPORT OUTPUT FILES ───────────────────────────────────────────────────────

input_files = (
    glob.glob(f'{folder_paths["input_agent_activity"]}/*.csv') +
    glob.glob(f'{folder_paths["input_agent_activity"]}/*.xlsx')
)
valid_weeks = {Path(f).stem for f in input_files}
print(sorted(valid_weeks))

for week_val, group in RTA_REPORT.group_by('Week_Monday'):
    wk = week_val[0]
    if wk in valid_weeks:
        group.write_csv(os.path.join(folder_paths['output_rta_all'], f'{wk}.csv'))
    else:
        print(f'Bỏ qua RTA_REPORT tuần {wk} (spillover ca đêm Chủ Nhật).')

for week_val, group in RTA_EXTEND.group_by('Week_Monday'):
    wk = week_val[0]
    if wk in valid_weeks:
        group.write_csv(os.path.join(folder_paths['rta_intervals_output'], f'{wk}.csv'))
    else:
        print(f'Bỏ qua RTA_EXTEND tuần {wk}.')

['2026-01-05', '2026-01-12', '2026-01-19', '2026-01-26', '2026-02-02', '2026-02-09', '2026-02-16', '2026-02-23', '2026-03-02', '2026-03-09', '2026-03-16', '2026-03-23', '2026-03-30', '2026-04-06', '2026-04-13', '2026-04-20', '2026-04-27', '2026-05-04', '2026-05-11']
Bỏ qua RTA_REPORT tuần 2026-05-18 (spillover ca đêm Chủ Nhật).
Bỏ qua RTA_REPORT tuần None (spillover ca đêm Chủ Nhật).
Bỏ qua RTA_EXTEND tuần None.
